In [27]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, confusion_matrix
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
import joblib
import json
from datetime import datetime
import warnings
import os
warnings.filterwarnings('ignore')
%run DeepProject.ipynb

LSTM DATASET1

In [28]:
df = pd.read_csv('usgs_main.csv')
df1 = df.copy()
df1['time'] = pd.to_datetime(df1['time'])
df1 = df1.sort_values('time')  # Kronolojik sıralama
df1 = df1.dropna(subset=['latitude', 'longitude', 'depth', 'mag', 'time'])    
df1['lats'] = np.floor(df1['latitude']).astype(int)
df1['lons'] = np.floor(df1['longitude']).astype(int)
dfdeep=df1.set_index('time').resample('D').apply({
    'mag':'mean',
    'latitude':'mean',
    'longitude':'mean',
    'depth':'mean'
})
dfdeep = dfdeep.reset_index(drop=True)
dfdeep.index = dfdeep.index + 1
dfdeep.index.name = 'timeindex'
dfdeep['futuremag'] = dfdeep['mag'].shift(-1)
dfdeep['futuredepth'] = dfdeep['depth'].shift(-1)
dfdeep['futurelat'] = dfdeep['latitude'].shift(-1)
dfdeep['futurelon'] = dfdeep['longitude'].shift(-1)
dfdeep = dfdeep.dropna(subset=['futuremag', 'futuredepth', 'futurelat', 'futurelon'])

In [29]:
train_size = int(0.8 * len(dfdeep))
train_data = dfdeep[:train_size]
test_data = dfdeep[train_size:]




In [30]:
model = SimpleLSTMEarthquakePredictor(
    random_state=42,
    use_optuna=True,
    n_trials=20  # 20 farklı hiperparametre kombinasyonu denenecek
)
model.fit(train_data)
print(f" En iyi hiperparametreler kullanılıyor: {model.best_params}")
test_predictions = model.predict(test_data)

[I 2025-07-27 20:36:56,153] A new study created in memory with name: no-name-f7f697dd-dc71-4cc6-b5a5-6c8ccc311127
[I 2025-07-27 20:36:59,136] Trial 0 finished with value: 0.02364904937985775 and parameters: {'sequence_length': 10, 'lstm_units': 32, 'lstm_layers': 1, 'dropout_rate': 0.12323344486727979, 'dense_dim': 16, 'learning_rate': 0.008706020878304856, 'batch_size': 16, 'optimizer': 'rmsprop', 'activation': 'relu'}. Best is trial 0 with value: 0.02364904937985775.
[I 2025-07-27 20:37:03,166] Trial 1 finished with value: 0.02741580163866841 and parameters: {'sequence_length': 9, 'lstm_units': 32, 'lstm_layers': 2, 'dropout_rate': 0.41407038455720546, 'dense_dim': 64, 'learning_rate': 0.0016409286730647919, 'batch_size': 64, 'optimizer': 'adam', 'activation': 'relu'}. Best is trial 0 with value: 0.02364904937985775.
[I 2025-07-27 20:37:10,398] Trial 2 finished with value: 0.06515366889785933 and parameters: {'sequence_length': 15, 'lstm_units': 128, 'lstm_layers': 3, 'dropout_rate':

En iyi parametreler: {'sequence_length': 20, 'lstm_units': 128, 'lstm_layers': 3, 'dropout_rate': 0.10718475024592794, 'dense_dim': 128, 'learning_rate': 0.00011952270129143879, 'batch_size': 64, 'optimizer': 'rmsprop', 'activation': 'tanh'}
 En iyi hiperparametreler kullanılıyor: {'sequence_length': 20, 'lstm_units': 128, 'lstm_layers': 3, 'dropout_rate': 0.10718475024592794, 'dense_dim': 128, 'learning_rate': 0.00011952270129143879, 'batch_size': 64, 'optimizer': 'rmsprop', 'activation': 'tanh'}


In [31]:
test_actual = test_data[['futuremag', 'futuredepth', 'futurelat', 'futurelon']].dropna()
min_len = min(len(test_predictions), len(test_actual))
# Her target için detaylı metrikler
target_names = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
target_labels = ['Magnitude', 'Depth', 'Latitude', 'Longitude']

In [32]:
for i, (name, label) in enumerate(zip(target_names, target_labels)):
    if i < test_predictions.shape[1] and i < test_actual.shape[1]:
        target_mse = mean_squared_error(
            test_actual.iloc[:min_len, i], 
            test_predictions[:min_len, i]
        )
        target_mae = mean_absolute_error(
            test_actual.iloc[:min_len, i], 
            test_predictions[:min_len, i]
        )
        target_r2 = r2_score(
            test_actual.iloc[:min_len, i], 
            test_predictions[:min_len, i]
        )
        
        print(f"{label:<12}: MSE={target_mse:.3f}, MAE={target_mae:.3f}, R²={target_r2:.3f}")

Magnitude   : MSE=0.035, MAE=0.147, R²=-0.491
Depth       : MSE=29.165, MAE=3.862, R²=-0.145
Latitude    : MSE=4.407, MAE=1.549, R²=-0.031
Longitude   : MSE=34.867, MAE=4.591, R²=-0.179


EŞİĞE GÖRE TAHMİN(ML DEKİ SİSTEMLE AYNI,AYNI KOD KULLANILDI)

In [33]:
#AYNI KODU KULLANDIK,DEĞERLENDİRME İÇİN
magnitude_thresholds = [1.7,2.0, 2.5, 3.0, 3.5, 4.0]

for threshold in magnitude_thresholds:
    print(f"\nBüyüklük Eşiği: {threshold}")
    
    # Tahmin edilen ve gerçek büyüklükler
    predicted_magnitudes = test_predictions[:min_len, 0]  
    actual_magnitudes = test_actual.iloc[:min_len, 0].values
    
    # Eşiğe göre ayır
    predicted_earthquake = (predicted_magnitudes >= threshold).astype(int)
    actual_earthquake = (actual_magnitudes >= threshold).astype(int)
    accuracy = (predicted_earthquake == actual_earthquake).mean()
    
    # Detaylı istatistikler
    total_samples = len(actual_earthquake)
    actual_earthquakes = actual_earthquake.sum()
    predicted_earthquakes = predicted_earthquake.sum()
    correct_predictions = (predicted_earthquake == actual_earthquake).sum()
    
    print(f"Toplam örnek sayısı: {total_samples}")
    print(f"Gerçek deprem sayısı (>={threshold}): {actual_earthquakes}")
    print(f"Tahmin edilen deprem sayısı (>={threshold}): {predicted_earthquakes}")
    print(f"Doğru tahmin sayısı: {correct_predictions}")
    print(f"Doğruluk (Accuracy): {accuracy:.3f}")
    
    # Confusion matrix güvenli hesaplama,try except de yapay zeka yardımı gerekti
    try:
        cm = confusion_matrix(actual_earthquake, predicted_earthquake)
        if cm.size == 4:  # 2x2 matrix
            tn, fp, fn, tp = cm.ravel()
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            
            print(f"Kesinlik (Precision): {precision:.3f}")
            print(f"Duyarlılık (Recall): {recall:.3f}")
            print(f"F1-Score: {f1_score:.3f}")
            print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
        else:
            print("Tek sınıf mevcut, detaylı metrikler hesaplanamıyor")
    except:
        print("Confusion matrix hesaplanamıyor")


Büyüklük Eşiği: 1.7
Toplam örnek sayısı: 37
Gerçek deprem sayısı (>=1.7): 17
Tahmin edilen deprem sayısı (>=1.7): 0
Doğru tahmin sayısı: 20
Doğruluk (Accuracy): 0.541
Kesinlik (Precision): 0.000
Duyarlılık (Recall): 0.000
F1-Score: 0.000
Confusion Matrix: TN=20, FP=0, FN=17, TP=0

Büyüklük Eşiği: 2.0
Toplam örnek sayısı: 37
Gerçek deprem sayısı (>=2.0): 2
Tahmin edilen deprem sayısı (>=2.0): 0
Doğru tahmin sayısı: 35
Doğruluk (Accuracy): 0.946
Kesinlik (Precision): 0.000
Duyarlılık (Recall): 0.000
F1-Score: 0.000
Confusion Matrix: TN=35, FP=0, FN=2, TP=0

Büyüklük Eşiği: 2.5
Toplam örnek sayısı: 37
Gerçek deprem sayısı (>=2.5): 0
Tahmin edilen deprem sayısı (>=2.5): 0
Doğru tahmin sayısı: 37
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 3.0
Toplam örnek sayısı: 37
Gerçek deprem sayısı (>=3.0): 0
Tahmin edilen deprem sayısı (>=3.0): 0
Doğru tahmin sayısı: 37
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

B

KONUMA GÖRE TAHMİN

In [34]:
test_with_predictions = test_data.iloc[:min_len].copy()
test_with_predictions['predicted_mag'] = test_predictions[:min_len, 0]
test_with_predictions['predicted_lat'] = test_predictions[:min_len, 2]
test_with_predictions['predicted_lon'] = test_predictions[:min_len, 3]

# Daha büyük bölge grupları oluştur (1.0 derece aralıklarla)
test_with_predictions['lat_group'] = np.round(test_with_predictions['latitude'])
test_with_predictions['lon_group'] = np.round(test_with_predictions['longitude'])
test_with_predictions['location_group'] = test_with_predictions['lat_group'].astype(str) + '_' + test_with_predictions['lon_group'].astype(str)

# Her konum grubu için değerlendirme
location_groups = test_with_predictions.groupby('location_group').size()
valid_locations = location_groups[location_groups >= 1].index

print(f"Yeterli veri olan bölge sayısı: {len(valid_locations)}")
count=0
threshold =1.6

for location in valid_locations[:10]:
    location_data = test_with_predictions[test_with_predictions['location_group'] == location]
    lat, lon = location.split('_')
    
    # threshold'dan büyük deprem var mı bakıyoruz,sondaki analiz yapay zek yardımıyla yazıldı
    has_actual_earthquake = (location_data['futuremag'] >= threshold).any()
    has_predicted_earthquake = (location_data['predicted_mag'] >= threshold).any()
    correct_prediction = has_actual_earthquake == has_predicted_earthquake
    if correct_prediction:
        count+=1
    print(f"Bölge ({lat}°, {lon}°) - Veri sayısı: {len(location_data)}")
    print(f"  Gerçek: {'Deprem var' if has_actual_earthquake else 'Deprem yok'}")
    print(f"  Tahmin: {'Deprem var' if has_predicted_earthquake else 'Deprem yok'}")
    print(f"  Doğru tahmin: {'✓' if correct_prediction else '✗'}")
    print(f"  Max gerçek büyüklük: {location_data['futuremag'].max():.2f}")
    print(f"  Max tahmin büyüklük: {location_data['predicted_mag'].max():.2f}")
    print()
print(f"Doğru tahmin oranı{(count/len(valid_locations))}")
    

Yeterli veri olan bölge sayısı: 32
Bölge (32.0°, -112.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem var
  Doğru tahmin: ✗
  Max gerçek büyüklük: 1.41
  Max tahmin büyüklük: 1.61

Bölge (33.0°, -110.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.91
  Max tahmin büyüklük: 1.61

Bölge (35.0°, -114.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 2.02
  Max tahmin büyüklük: 1.62

Bölge (36.0°, -100.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem var
  Doğru tahmin: ✗
  Max gerçek büyüklük: 1.60
  Max tahmin büyüklük: 1.62

Bölge (36.0°, -107.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.95
  Max tahmin büyüklük: 1.62

Bölge (36.0°, -117.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.62
  Max tahmin büyüklük: 1.61

Bölge (37.0°, -110.0°) 

kayıt

In [35]:
model.model.save('models/simpleLSTMdataset1.keras')
lstm_best_params = {
    'best_params': model.best_params
}
with open('models/simpleLSTMdataset1.keras.json', 'w') as f:
    json.dump(lstm_best_params, f, indent=2)

LSTM+ATTENTİON-DATASET1

In [36]:
#daha çok deneme,outlier detect ve attention 
model2=AdvancedLSTMEarthquakePredictor(
    random_state=42,
    use_optuna=True,
    use_attention=True,
    n_trials=25
)
model2.fit(train_data)
print(f" En iyi hiperparametreler kullanılıyor: {model2.best_params}")
test_predictions = model2.predict(test_data)

[I 2025-07-27 20:39:04,947] A new study created in memory with name: no-name-b6b3dd05-b66e-4a51-9532-30851151b842
[I 2025-07-27 20:39:09,203] Trial 0 finished with value: 0.02323912840992729 and parameters: {'sequence_length': 10, 'lstm_units': 32, 'lstm_layers': 1, 'dropout_rate': 0.12323344486727979, 'dense_dim': 16, 'learning_rate': 0.008706020878304856, 'batch_size': 16, 'optimizer': 'rmsprop', 'activation': 'relu'}. Best is trial 0 with value: 0.02323912840992729.
[I 2025-07-27 20:39:13,997] Trial 1 finished with value: 0.02760874362019174 and parameters: {'sequence_length': 9, 'lstm_units': 32, 'lstm_layers': 2, 'dropout_rate': 0.41407038455720546, 'dense_dim': 64, 'learning_rate': 0.0016409286730647919, 'batch_size': 64, 'optimizer': 'adam', 'activation': 'relu'}. Best is trial 0 with value: 0.02323912840992729.
[I 2025-07-27 20:39:21,431] Trial 2 finished with value: 0.2207749439123579 and parameters: {'sequence_length': 15, 'lstm_units': 128, 'lstm_layers': 3, 'dropout_rate': 

En iyi parametreler: {'sequence_length': 19, 'lstm_units': 256, 'lstm_layers': 2, 'dropout_rate': 0.30012324171230487, 'dense_dim': 128, 'learning_rate': 0.001827900125554867, 'batch_size': 64, 'optimizer': 'rmsprop', 'activation': 'tanh'}
 En iyi hiperparametreler kullanılıyor: {'sequence_length': 19, 'lstm_units': 256, 'lstm_layers': 2, 'dropout_rate': 0.30012324171230487, 'dense_dim': 128, 'learning_rate': 0.001827900125554867, 'batch_size': 64, 'optimizer': 'rmsprop', 'activation': 'tanh'}


In [37]:
test_actual = test_data[['futuremag', 'futuredepth', 'futurelat', 'futurelon']].dropna()
min_len = min(len(test_predictions), len(test_actual))
# Her target için detaylı metrikler
target_names = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
target_labels = ['Magnitude', 'Depth', 'Latitude', 'Longitude']
for i, (name, label) in enumerate(zip(target_names, target_labels)):
    if i < test_predictions.shape[1] and i < test_actual.shape[1]:
        target_mse = mean_squared_error(
            test_actual.iloc[:min_len, i], 
            test_predictions[:min_len, i]
        )
        target_mae = mean_absolute_error(
            test_actual.iloc[:min_len, i], 
            test_predictions[:min_len, i]
        )
        target_r2 = r2_score(
            test_actual.iloc[:min_len, i], 
            test_predictions[:min_len, i]
        )
        
        print(f"{label:<12}: MSE={target_mse:.3f}, MAE={target_mae:.3f}, R²={target_r2:.3f}")

Magnitude   : MSE=0.035, MAE=0.147, R²=-0.468
Depth       : MSE=28.739, MAE=3.872, R²=-0.147
Latitude    : MSE=4.256, MAE=1.513, R²=-0.023
Longitude   : MSE=33.767, MAE=4.480, R²=-0.161


EŞİKSEL BAKIŞ(Birebir aynı kod,aradaki farkı net görürüz)

In [38]:
#yine aynı kod
magnitude_thresholds = [1.7,2.0, 2.5, 3.0, 3.5, 4.0]

for threshold in magnitude_thresholds:
    print(f"\nBüyüklük Eşiği: {threshold}")
    
    # Tahmin edilen ve gerçek büyüklükler
    predicted_magnitudes = test_predictions[:min_len, 0]  
    actual_magnitudes = test_actual.iloc[:min_len, 0].values
    
    # Eşiğe göre ayır
    predicted_earthquake = (predicted_magnitudes >= threshold).astype(int)
    actual_earthquake = (actual_magnitudes >= threshold).astype(int)
    accuracy = (predicted_earthquake == actual_earthquake).mean()
    
    # Detaylı istatistikler
    total_samples = len(actual_earthquake)
    actual_earthquakes = actual_earthquake.sum()
    predicted_earthquakes = predicted_earthquake.sum()
    correct_predictions = (predicted_earthquake == actual_earthquake).sum()
    
    print(f"Toplam örnek sayısı: {total_samples}")
    print(f"Gerçek deprem sayısı (>={threshold}): {actual_earthquakes}")
    print(f"Tahmin edilen deprem sayısı (>={threshold}): {predicted_earthquakes}")
    print(f"Doğru tahmin sayısı: {correct_predictions}")
    print(f"Doğruluk (Accuracy): {accuracy:.3f}")
    
    # Confusion matrix güvenli hesaplama,try except de yapay zeka yardımı gerekti
    try:
        cm = confusion_matrix(actual_earthquake, predicted_earthquake)
        if cm.size == 4:  # 2x2 matrix
            tn, fp, fn, tp = cm.ravel()
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            
            print(f"Kesinlik (Precision): {precision:.3f}")
            print(f"Duyarlılık (Recall): {recall:.3f}")
            print(f"F1-Score: {f1_score:.3f}")
            print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
        else:
            print("Tek sınıf mevcut, detaylı metrikler hesaplanamıyor")
    except:
        print("Confusion matrix hesaplanamıyor")


Büyüklük Eşiği: 1.7
Toplam örnek sayısı: 38
Gerçek deprem sayısı (>=1.7): 18
Tahmin edilen deprem sayısı (>=1.7): 0
Doğru tahmin sayısı: 20
Doğruluk (Accuracy): 0.526
Kesinlik (Precision): 0.000
Duyarlılık (Recall): 0.000
F1-Score: 0.000
Confusion Matrix: TN=20, FP=0, FN=18, TP=0

Büyüklük Eşiği: 2.0
Toplam örnek sayısı: 38
Gerçek deprem sayısı (>=2.0): 2
Tahmin edilen deprem sayısı (>=2.0): 0
Doğru tahmin sayısı: 36
Doğruluk (Accuracy): 0.947
Kesinlik (Precision): 0.000
Duyarlılık (Recall): 0.000
F1-Score: 0.000
Confusion Matrix: TN=36, FP=0, FN=2, TP=0

Büyüklük Eşiği: 2.5
Toplam örnek sayısı: 38
Gerçek deprem sayısı (>=2.5): 0
Tahmin edilen deprem sayısı (>=2.5): 0
Doğru tahmin sayısı: 38
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 3.0
Toplam örnek sayısı: 38
Gerçek deprem sayısı (>=3.0): 0
Tahmin edilen deprem sayısı (>=3.0): 0
Doğru tahmin sayısı: 38
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

B

KONUMSAL BAKIŞ(AYNI KODLA NET PERFORMANS KARŞILAŞTIRMASI)

In [39]:
test_with_predictions = test_data.iloc[:min_len].copy()
test_with_predictions['predicted_mag'] = test_predictions[:min_len, 0]
test_with_predictions['predicted_lat'] = test_predictions[:min_len, 2]
test_with_predictions['predicted_lon'] = test_predictions[:min_len, 3]

# Daha büyük bölge grupları oluştur (1.0 derece aralıklarla)
test_with_predictions['lat_group'] = np.round(test_with_predictions['latitude'])
test_with_predictions['lon_group'] = np.round(test_with_predictions['longitude'])
test_with_predictions['location_group'] = test_with_predictions['lat_group'].astype(str) + '_' + test_with_predictions['lon_group'].astype(str)

# Her konum grubu için değerlendirme
location_groups = test_with_predictions.groupby('location_group').size()
valid_locations = location_groups[location_groups >= 1].index

print(f"Yeterli veri olan bölge sayısı: {len(valid_locations)}")
count=0
threshold =1.6

for location in valid_locations[:10]:
    location_data = test_with_predictions[test_with_predictions['location_group'] == location]
    lat, lon = location.split('_')
    
    # threshold'dan büyük deprem var mı bakıyoruz,sondaki analiz yapay zek yardımıyla yazıldı
    has_actual_earthquake = (location_data['futuremag'] >= threshold).any()
    has_predicted_earthquake = (location_data['predicted_mag'] >= threshold).any()
    correct_prediction = has_actual_earthquake == has_predicted_earthquake
    if correct_prediction:
        count+=1
    print(f"Bölge ({lat}°, {lon}°) - Veri sayısı: {len(location_data)}")
    print(f"  Gerçek: {'Deprem var' if has_actual_earthquake else 'Deprem yok'}")
    print(f"  Tahmin: {'Deprem var' if has_predicted_earthquake else 'Deprem yok'}")
    print(f"  Doğru tahmin: {'✓' if correct_prediction else '✗'}")
    print(f"  Max gerçek büyüklük: {location_data['futuremag'].max():.2f}")
    print(f"  Max tahmin büyüklük: {location_data['predicted_mag'].max():.2f}")
    print()
print(f"Doğru tahmin oranı{(count/len(valid_locations))}")
    

Yeterli veri olan bölge sayısı: 33
Bölge (32.0°, -112.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem var
  Doğru tahmin: ✗
  Max gerçek büyüklük: 1.41
  Max tahmin büyüklük: 1.62

Bölge (33.0°, -110.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.91
  Max tahmin büyüklük: 1.62

Bölge (35.0°, -103.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.94
  Max tahmin büyüklük: 1.62

Bölge (35.0°, -114.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 2.02
  Max tahmin büyüklük: 1.62

Bölge (36.0°, -100.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem var
  Doğru tahmin: ✗
  Max gerçek büyüklük: 1.60
  Max tahmin büyüklük: 1.62

Bölge (36.0°, -107.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.95
  Max tahmin büyüklük: 1.62

Bölge (36.0°, -117.0°) 

In [40]:
model2.model.save('models/LSTMAttentiondataset1.keras')
lstm_best_params = {
    'best_params': model2.best_params
}
with open('models/LSTMAttentiondataset1.keras.json', 'w') as f:
    json.dump(lstm_best_params, f, indent=2)

DATASET2-UFAK DEĞİŞİKLİKLERLE UYGULANACAK

In [41]:
#ML deki gibi preprocess
nx=pd.read_csv('Significant Earthquake Dataset 1900-2023.csv')
dfls=nx
dfls=dfls.rename(columns={'Time':'time','Mag':'mag','Depth':'depth','Latitude':'latitude','Longitude':'longitude' })
dfls['time'] = pd.to_datetime(dfls['time'])
dfls = dfls.sort_values('time')
dfls = dfls.dropna(subset=['latitude','longitude','depth','mag','time'])
dfls['lats'] = np.floor(dfls['latitude']).astype(int)
dfls['lons'] = np.floor(dfls['longitude']).astype(int)
dfother = dfls.set_index('time').resample('YE').apply({
    'mag':'mean',
    'latitude':'mean',
    'longitude':'mean',
    'depth':'mean',
})
dfother = dfother.reset_index(drop=True)
dfother.index = dfother.index + 1
dfother.index.name = 'timeindex'
dfother['futuremag']=dfother['mag'].shift(-1)
dfother['futuredepth']=dfother['depth'].shift(-1)
dfother['futurelat']=dfother['latitude'].shift(-1)
dfother['futurelon']=dfother['longitude'].shift(-1)
dfother = dfother.dropna(subset=['futuremag', 'futuredepth', 'futurelat', 'futurelon'])

In [42]:
train_size = int(0.8 * len(dfother))
train_data = dfother[:train_size]
test_data = dfother[train_size:]


In [43]:
model3 = SimpleLSTMEarthquakePredictor(
    random_state=42,
    use_optuna=True,
    n_trials=22
)
model3.fit(train_data)
print(f" En iyi hiperparametreler kullanılıyor: {model3.best_params}")
test_predictions = model3.predict(test_data)

[I 2025-07-27 20:41:35,992] A new study created in memory with name: no-name-afdf42df-579c-4e2e-a737-ba882b4cea57
[I 2025-07-27 20:41:39,124] Trial 0 finished with value: 0.08021725544207893 and parameters: {'sequence_length': 10, 'lstm_units': 32, 'lstm_layers': 1, 'dropout_rate': 0.12323344486727979, 'dense_dim': 16, 'learning_rate': 0.008706020878304856, 'batch_size': 16, 'optimizer': 'rmsprop', 'activation': 'relu'}. Best is trial 0 with value: 0.08021725544207893.
[I 2025-07-27 20:41:43,323] Trial 1 finished with value: 0.012652648888151966 and parameters: {'sequence_length': 9, 'lstm_units': 32, 'lstm_layers': 2, 'dropout_rate': 0.41407038455720546, 'dense_dim': 64, 'learning_rate': 0.0016409286730647919, 'batch_size': 64, 'optimizer': 'adam', 'activation': 'relu'}. Best is trial 1 with value: 0.012652648888151966.
[I 2025-07-27 20:41:48,839] Trial 2 finished with value: 0.0004469307307478294 and parameters: {'sequence_length': 15, 'lstm_units': 128, 'lstm_layers': 3, 'dropout_ra

En iyi parametreler: {'sequence_length': 15, 'lstm_units': 128, 'lstm_layers': 3, 'dropout_rate': 0.20351199264000677, 'dense_dim': 16, 'learning_rate': 0.00023426581058204064, 'batch_size': 16, 'optimizer': 'adam', 'activation': 'relu'}
 En iyi hiperparametreler kullanılıyor: {'sequence_length': 15, 'lstm_units': 128, 'lstm_layers': 3, 'dropout_rate': 0.20351199264000677, 'dense_dim': 16, 'learning_rate': 0.00023426581058204064, 'batch_size': 16, 'optimizer': 'adam', 'activation': 'relu'}


In [44]:
test_actual = test_data[['futuremag', 'futuredepth', 'futurelat', 'futurelon']].dropna()
min_len = min(len(test_predictions), len(test_actual))
target_names = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
target_labels = ['Magnitude', 'Depth', 'Latitude', 'Longitude']
for i, (name, label) in enumerate(zip(target_names, target_labels)):
    if i < test_predictions.shape[1] and i < test_actual.shape[1]:
        target_mse = mean_squared_error(
            test_actual.iloc[:min_len, i], 
            test_predictions[:min_len, i]
        )
        target_mae = mean_absolute_error(
            test_actual.iloc[:min_len, i], 
            test_predictions[:min_len, i]
        )
        target_r2 = r2_score(
            test_actual.iloc[:min_len, i], 
            test_predictions[:min_len, i]
        )
        
        print(f"{label:<12}: MSE={target_mse:.3f}, MAE={target_mae:.3f}, R²={target_r2:.3f}")

Magnitude   : MSE=0.011, MAE=0.106, R²=-35.877
Depth       : MSE=632.121, MAE=22.029, R²=-3.311
Latitude    : MSE=43.746, MAE=6.266, R²=-8.743
Longitude   : MSE=106.781, MAE=8.998, R²=-0.705


EŞİK

In [45]:
magnitude_thresholds = [5.8,6.0, 6.5, 7.0, 7.5]
for threshold in magnitude_thresholds:
    print(f"\nBüyüklük Eşiği: {threshold}")
    
    
    predicted_magnitudes = test_predictions[:min_len, 0]  
    actual_magnitudes = test_actual.iloc[:min_len, 0].values
    
    
    predicted_earthquake = (predicted_magnitudes >= threshold).astype(int)
    actual_earthquake = (actual_magnitudes >= threshold).astype(int)
    accuracy = (predicted_earthquake == actual_earthquake).mean()
    
    
    total_samples = len(actual_earthquake)
    actual_earthquakes = actual_earthquake.sum()
    predicted_earthquakes = predicted_earthquake.sum()
    correct_predictions = (predicted_earthquake == actual_earthquake).sum()
    
    print(f"Toplam örnek sayısı: {total_samples}")
    print(f"Gerçek deprem sayısı (>={threshold}): {actual_earthquakes}")
    print(f"Tahmin edilen deprem sayısı (>={threshold}): {predicted_earthquakes}")
    print(f"Doğru tahmin sayısı: {correct_predictions}")
    print(f"Doğruluk (Accuracy): {accuracy:.3f}")
    
    
    try:
        cm = confusion_matrix(actual_earthquake, predicted_earthquake)
        if cm.size == 4:  # 2x2 matrix
            tn, fp, fn, tp = cm.ravel()
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            
            print(f"Kesinlik (Precision): {precision:.3f}")
            print(f"Duyarlılık (Recall): {recall:.3f}")
            print(f"F1-Score: {f1_score:.3f}")
            print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
        else:
            print("Tek sınıf mevcut, detaylı metrikler hesaplanamıyor")
    except:
        print("Confusion matrix hesaplanamıyor")


Büyüklük Eşiği: 5.8
Toplam örnek sayısı: 9
Gerçek deprem sayısı (>=5.8): 9
Tahmin edilen deprem sayısı (>=5.8): 0
Doğru tahmin sayısı: 0
Doğruluk (Accuracy): 0.000
Kesinlik (Precision): 0.000
Duyarlılık (Recall): 0.000
F1-Score: 0.000
Confusion Matrix: TN=0, FP=0, FN=9, TP=0

Büyüklük Eşiği: 6.0
Toplam örnek sayısı: 9
Gerçek deprem sayısı (>=6.0): 0
Tahmin edilen deprem sayısı (>=6.0): 0
Doğru tahmin sayısı: 9
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 6.5
Toplam örnek sayısı: 9
Gerçek deprem sayısı (>=6.5): 0
Tahmin edilen deprem sayısı (>=6.5): 0
Doğru tahmin sayısı: 9
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 7.0
Toplam örnek sayısı: 9
Gerçek deprem sayısı (>=7.0): 0
Tahmin edilen deprem sayısı (>=7.0): 0
Doğru tahmin sayısı: 9
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 7.5
Toplam örnek sayısı: 9
Gerçek deprem sayısı (>=7.5): 0

KONUM(ML deki gibi,metrik kodları aynı olacak)

In [46]:
test_with_predictions_2 = test_data.iloc[:min_len].copy()
test_with_predictions_2['predicted_mag'] = test_predictions[:min_len, 0]
test_with_predictions_2['predicted_lat'] = test_predictions[:min_len, 2]
test_with_predictions_2['predicted_lon'] = test_predictions[:min_len, 3]


test_with_predictions_2['lat_group'] = np.round(test_with_predictions_2['latitude'] / 5.0) * 5.0
test_with_predictions_2['lon_group'] = np.round(test_with_predictions_2['longitude'] / 5.0) * 5.0
test_with_predictions_2['location_group'] = test_with_predictions_2['lat_group'].astype(str) + '_' + test_with_predictions_2['lon_group'].astype(str)


location_groups_2 = test_with_predictions_2.groupby('location_group').size()
valid_locations_2 = location_groups_2[location_groups_2 >= 1].index 

print(f"Yeterli veri olan bölge sayısı: {len(valid_locations_2)}")

threshold = 5.9 
count=0
for location in valid_locations_2[:10]:
    location_data = test_with_predictions_2[test_with_predictions_2['location_group'] == location]
    lat, lon = location.split('_')
    has_actual_earthquake = (location_data['futuremag'] >= threshold).any()
    has_predicted_earthquake = (location_data['predicted_mag'] >= threshold).any()
    correct_prediction = has_actual_earthquake == has_predicted_earthquake
    if correct_prediction:
        count+=1
    print(f"Bölge ({lat}°, {lon}°) - Veri sayısı: {len(location_data)}")
    print(f"  Gerçek: {'Deprem var' if has_actual_earthquake else 'Deprem yok'}")
    print(f"  Tahmin: {'Deprem var' if has_predicted_earthquake else 'Deprem yok'}")
    print(f"  Doğru tahmin: {'✓' if correct_prediction else '✗'}")
    print(f"  Max gerçek büyüklük: {location_data['futuremag'].max():.2f}")
    print(f"  Max tahmin büyüklük: {location_data['predicted_mag'].max():.2f}")
    print()
print(f"accuracy: {count/len(valid_locations_2)}")

Yeterli veri olan bölge sayısı: 8
Bölge (-0.0°, 25.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.87
  Max tahmin büyüklük: 5.77

Bölge (-0.0°, 40.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.86
  Max tahmin büyüklük: 5.77

Bölge (-0.0°, 55.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.88
  Max tahmin büyüklük: 5.77

Bölge (-5.0°, 45.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem yok
  Doğru tahmin: ✗
  Max gerçek büyüklük: 5.91
  Max tahmin büyüklük: 5.77

Bölge (0.0°, 40.0°) - Veri sayısı: 2
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.89
  Max tahmin büyüklük: 5.77

Bölge (0.0°, 45.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.86
  Max tahmin büyüklük: 5.77

Bölge (5.0°, 40.0°) - Veri sayısı: 1
 

In [47]:
model3.model.save('models/simpleLSTMdataset2.keras')
lstm_best_params = {
    'best_params': model3.best_params
}
with open('models/simpleLSTMdataset2.keras.json', 'w') as f:
    json.dump(lstm_best_params, f, indent=2)

LSTM+ATTENTION DATASET2

In [48]:
model4=AdvancedLSTMEarthquakePredictor(
    random_state=42,
    use_optuna=True,
    use_attention=True,
    n_trials=26
)
model4.fit(train_data)
print(f" En iyi hiperparametreler kullanılıyor: {model4.best_params}")
test_predictions = model4.predict(test_data)

[I 2025-07-27 20:43:28,355] A new study created in memory with name: no-name-36fe7c11-f138-478b-a7f2-55b3831f97b2
[I 2025-07-27 20:43:31,699] Trial 0 finished with value: 0.07851428887955948 and parameters: {'sequence_length': 10, 'lstm_units': 32, 'lstm_layers': 1, 'dropout_rate': 0.12323344486727979, 'dense_dim': 16, 'learning_rate': 0.008706020878304856, 'batch_size': 16, 'optimizer': 'rmsprop', 'activation': 'relu'}. Best is trial 0 with value: 0.07851428887955948.
[I 2025-07-27 20:43:36,174] Trial 1 finished with value: 0.0741899981651975 and parameters: {'sequence_length': 9, 'lstm_units': 32, 'lstm_layers': 2, 'dropout_rate': 0.41407038455720546, 'dense_dim': 64, 'learning_rate': 0.0016409286730647919, 'batch_size': 64, 'optimizer': 'adam', 'activation': 'relu'}. Best is trial 1 with value: 0.0741899981651975.
[I 2025-07-27 20:43:42,520] Trial 2 finished with value: 0.01896834083241155 and parameters: {'sequence_length': 15, 'lstm_units': 128, 'lstm_layers': 3, 'dropout_rate': 0

En iyi parametreler: {'sequence_length': 10, 'lstm_units': 32, 'lstm_layers': 2, 'dropout_rate': 0.4887128330883843, 'dense_dim': 16, 'learning_rate': 0.00037126241790405324, 'batch_size': 32, 'optimizer': 'rmsprop', 'activation': 'relu'}
 En iyi hiperparametreler kullanılıyor: {'sequence_length': 10, 'lstm_units': 32, 'lstm_layers': 2, 'dropout_rate': 0.4887128330883843, 'dense_dim': 16, 'learning_rate': 0.00037126241790405324, 'batch_size': 32, 'optimizer': 'rmsprop', 'activation': 'relu'}


In [49]:
test_actual = test_data[['futuremag', 'futuredepth', 'futurelat', 'futurelon']].dropna()
min_len = min(len(test_predictions), len(test_actual))
target_names = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
target_labels = ['Magnitude', 'Depth', 'Latitude', 'Longitude']
for i, (name, label) in enumerate(zip(target_names, target_labels)):
    if i < test_predictions.shape[1] and i < test_actual.shape[1]:
        target_mse = mean_squared_error(
            test_actual.iloc[:min_len, i], 
            test_predictions[:min_len, i]
        )
        target_mae = mean_absolute_error(
            test_actual.iloc[:min_len, i], 
            test_predictions[:min_len, i]
        )
        target_r2 = r2_score(
            test_actual.iloc[:min_len, i], 
            test_predictions[:min_len, i]
        )
        
        print(f"{label:<12}: MSE={target_mse:.3f}, MAE={target_mae:.3f}, R²={target_r2:.3f}")

Magnitude   : MSE=0.004, MAE=0.064, R²=-15.516
Depth       : MSE=886.244, MAE=27.878, R²=-6.769
Latitude    : MSE=12.166, MAE=2.378, R²=-0.008
Longitude   : MSE=1471.385, MAE=36.860, R²=-12.055


EŞİK KONTROLÜ

In [50]:
magnitude_thresholds = [5.8,6.0, 6.5, 7.0, 7.5]
for threshold in magnitude_thresholds:
    print(f"\nBüyüklük Eşiği: {threshold}")
    
    
    predicted_magnitudes = test_predictions[:min_len, 0]  
    actual_magnitudes = test_actual.iloc[:min_len, 0].values
    
    
    predicted_earthquake = (predicted_magnitudes >= threshold).astype(int)
    actual_earthquake = (actual_magnitudes >= threshold).astype(int)
    accuracy = (predicted_earthquake == actual_earthquake).mean()
    
    
    total_samples = len(actual_earthquake)
    actual_earthquakes = actual_earthquake.sum()
    predicted_earthquakes = predicted_earthquake.sum()
    correct_predictions = (predicted_earthquake == actual_earthquake).sum()
    
    print(f"Toplam örnek sayısı: {total_samples}")
    print(f"Gerçek deprem sayısı (>={threshold}): {actual_earthquakes}")
    print(f"Tahmin edilen deprem sayısı (>={threshold}): {predicted_earthquakes}")
    print(f"Doğru tahmin sayısı: {correct_predictions}")
    print(f"Doğruluk (Accuracy): {accuracy:.3f}")
    
    
    try:
        cm = confusion_matrix(actual_earthquake, predicted_earthquake)
        if cm.size == 4:  # 2x2 matrix
            tn, fp, fn, tp = cm.ravel()
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            
            print(f"Kesinlik (Precision): {precision:.3f}")
            print(f"Duyarlılık (Recall): {recall:.3f}")
            print(f"F1-Score: {f1_score:.3f}")
            print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
        else:
            print("Tek sınıf mevcut, detaylı metrikler hesaplanamıyor")
    except:
        print("Confusion matrix hesaplanamıyor")


Büyüklük Eşiği: 5.8
Toplam örnek sayısı: 14
Gerçek deprem sayısı (>=5.8): 14
Tahmin edilen deprem sayısı (>=5.8): 14
Doğru tahmin sayısı: 14
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 6.0
Toplam örnek sayısı: 14
Gerçek deprem sayısı (>=6.0): 0
Tahmin edilen deprem sayısı (>=6.0): 0
Doğru tahmin sayısı: 14
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 6.5
Toplam örnek sayısı: 14
Gerçek deprem sayısı (>=6.5): 0
Tahmin edilen deprem sayısı (>=6.5): 0
Doğru tahmin sayısı: 14
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 7.0
Toplam örnek sayısı: 14
Gerçek deprem sayısı (>=7.0): 0
Tahmin edilen deprem sayısı (>=7.0): 0
Doğru tahmin sayısı: 14
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 7.5
Toplam örnek sayısı: 14
Gerçek deprem sayısı (>=7.5): 0
Tahmin edilen deprem sayısı (>=7.5): 0
Doğru tahm

KONUMA GÖRE

In [51]:
test_with_predictions_2 = test_data.iloc[:min_len].copy()
test_with_predictions_2['predicted_mag'] = test_predictions[:min_len, 0]
test_with_predictions_2['predicted_lat'] = test_predictions[:min_len, 2]
test_with_predictions_2['predicted_lon'] = test_predictions[:min_len, 3]


test_with_predictions_2['lat_group'] = np.round(test_with_predictions_2['latitude'] / 5.0) * 5.0
test_with_predictions_2['lon_group'] = np.round(test_with_predictions_2['longitude'] / 5.0) * 5.0
test_with_predictions_2['location_group'] = test_with_predictions_2['lat_group'].astype(str) + '_' + test_with_predictions_2['lon_group'].astype(str)


location_groups_2 = test_with_predictions_2.groupby('location_group').size()
valid_locations_2 = location_groups_2[location_groups_2 >= 1].index 

print(f"Yeterli veri olan bölge sayısı: {len(valid_locations_2)}")

threshold = 5.9 
count=0
for location in valid_locations_2[:10]:
    location_data = test_with_predictions_2[test_with_predictions_2['location_group'] == location]
    lat, lon = location.split('_')
    has_actual_earthquake = (location_data['futuremag'] >= threshold).any()
    has_predicted_earthquake = (location_data['predicted_mag'] >= threshold).any()
    correct_prediction = has_actual_earthquake == has_predicted_earthquake
    if correct_prediction:
        count+=1
    print(f"Bölge ({lat}°, {lon}°) - Veri sayısı: {len(location_data)}")
    print(f"  Gerçek: {'Deprem var' if has_actual_earthquake else 'Deprem yok'}")
    print(f"  Tahmin: {'Deprem var' if has_predicted_earthquake else 'Deprem yok'}")
    print(f"  Doğru tahmin: {'✓' if correct_prediction else '✗'}")
    print(f"  Max gerçek büyüklük: {location_data['futuremag'].max():.2f}")
    print(f"  Max tahmin büyüklük: {location_data['predicted_mag'].max():.2f}")
    print()
print(f"accuracy: {count/len(valid_locations_2)}")

Yeterli veri olan bölge sayısı: 12
Bölge (-0.0°, 25.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem var
  Doğru tahmin: ✗
  Max gerçek büyüklük: 5.87
  Max tahmin büyüklük: 5.94

Bölge (-0.0°, 30.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.90
  Max tahmin büyüklük: 5.94

Bölge (-0.0°, 40.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem var
  Doğru tahmin: ✗
  Max gerçek büyüklük: 5.86
  Max tahmin büyüklük: 5.94

Bölge (-0.0°, 55.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem var
  Doğru tahmin: ✗
  Max gerçek büyüklük: 5.88
  Max tahmin büyüklük: 5.94

Bölge (-5.0°, 20.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem var
  Doğru tahmin: ✗
  Max gerçek büyüklük: 5.86
  Max tahmin büyüklük: 5.94

Bölge (-5.0°, 45.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.91
  Max tahmin büyüklük: 5.94

Bölge (0.0°, 40.0°) - Veri sayısı: 

In [52]:
model4.model.save('models/LSTMAttentiondataset2.keras')
lstm_best_params = {
    'best_params': model4.best_params
}
with open('models/sLSTMAttentiondataset2.keras.json', 'w') as f:
    json.dump(lstm_best_params, f, indent=2)